In [2]:
import os 
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
from PIL import Image

CLASSES = ['double_plant', 'drydown', 'endrow', 'nutrient_deficiency', 
           'planter_skip', 'storm_damage', 'water', 'waterway', 'weed_cluster']
class2id = {cls_name: i for i , cls_name in enumerate(CLASSES)}

BASE_DATA_DIR = r'D:\Developments of Cognify\datasets\Agriculture-Vision-2021'
SPLITS = ['train', 'val', 'test']

MODALITIES = ['rgb', 'nir']

# Флаг для применения CLAHE (вытягивание контраста) к изображениям. 
# Если True, то CLAHE будет применен к каждому изображению перед сохранением. 
# Это может помочь улучшить видимость деталей на изображениях, особенно в условиях плохого освещения или низкого контраста. 
# Если False, изображения будут сохранены без изменений.
APPLY_CLACHE = False

def apply_clahe(img):
    if img is None: 
        return None
    
    # Если изображение одноканальное 
    if len(img.shape) == 2 or img.shape[2] == 1:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(img)
    
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))

    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

for modality in MODALITIES:
    print(f"\n" + "-"*50)
    print(f"Запуск обработки модальности: {modality.upper()}")
    print('-'*50)

    for split in SPLITS:
        print(f"\n[{split.upper()}] начинаем обработку...")

        # Исходные пути
        DATA_DIR = os.path.join(BASE_DATA_DIR, split)
        IMG_DIR = os.path.join(DATA_DIR, 'images', modality)
        BOUND_DIR = os.path.join(DATA_DIR, 'boundaries')
        MASK_DIR = os.path.join(DATA_DIR, 'masks')
        LABELS_DIR = os.path.join(DATA_DIR, 'labels')

        # Пути сохранения в формате YOLO
        OUT_BASE_DIR = os.path.join(BASE_DATA_DIR, 'yolo_dataset', modality)
        OUT_IMAGES = os.path.join(OUT_BASE_DIR, 'images', split)
        OUT_LABELS = os.path.join(OUT_BASE_DIR, 'labels', split)

        os.makedirs(OUT_IMAGES, exist_ok=True)

        has_labels = os.path.exists(LABELS_DIR)
        if has_labels:
            os.makedirs(OUT_LABELS, exist_ok=True)
        else:
            print(f"[{split.upper()}] папка с разметкой (labels) не найдена. Будут обработаны только изображения.")
        
        image_paths = glob(os.path.join(IMG_DIR, '*.jpg')) + glob(os.path.join(IMG_DIR, '*.png'))

        if not image_paths:
            print(f"[{split.upper()}] изображения не найдены в {IMG_DIR}. Пропускаем.")
            continue

        for img_path in tqdm(image_paths, desc=f'Обработка {split}'):
            img_name = os.path.basename(img_path)
            base_name = os.path.splitext(img_name)[0]
            mask_name = base_name + '.png' 

            try:
                with Image.open(img_path) as img_pil:
                    icc_profile = img_pil.info.get('icc_profile')
                    img = np.array(img_pil)
            except Exception as e:
                print(f"Ошибка чтения {img_path}: {e}")
                continue 

            h, w = img.shape[:2]

            if APPLY_CLACHE:
                img = apply_clahe(img)

            bound_path = os.path.join(BOUND_DIR, mask_name)
            mask_path = os.path.join(MASK_DIR, mask_name)

            if os.path.exists(bound_path) and os.path.exists(mask_path):
                bound = cv2.imread(bound_path, cv2.IMREAD_GRAYSCALE)
                valid_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                combined_valid = cv2.bitwise_and(bound, valid_mask)
                
                # Создаем маску фона и заполняем её нейтральным серым цветом (128)
                background_mask = cv2.bitwise_not(combined_valid)
                mean_bg = np.full(img.shape, 128, dtype=np.uint8) 
                
                # Собираем изображение: валидная зона + нейтральный фон
                img_valid = cv2.bitwise_and(img, img, mask=combined_valid)
                img_bg = cv2.bitwise_and(mean_bg, mean_bg, mask=background_mask)
                img = cv2.add(img_valid, img_bg)

            out_pil = Image.fromarray(img)

            save_path = os.path.join(OUT_IMAGES, img_name)
            if icc_profile:
                out_pil.save(save_path, icc_profile=icc_profile)
            else:
                out_pil.save(save_path)

            # Обработка масок классов в полигоны 
            if has_labels:
                yolo_labels = []
                for cls_name in CLASSES:
                    cls_mask_path = os.path.join(LABELS_DIR, cls_name, mask_name)

                    if os.path.exists(cls_mask_path):
                        cls_mask = cv2.imread(cls_mask_path, cv2.IMREAD_GRAYSCALE)
                        if cls_mask is None or np.max(cls_mask) == 0:
                            continue 

                        # Убираем пиксельную зубчатость: легкое размытие + жесткий порог
                        blurred = cv2.GaussianBlur(cls_mask, (3, 3), 0)
                        _, thresh = cv2.threshold(blurred, 127, 255, cv2.THRESH_BINARY)

                        # Ищем контуры (CHAIN_APPROX_NONE сохраняет плавность геометрии)
                        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                        
                        for contour in contours:
                            # Оптимизируем число точек с сохранением геометрии (алгоритм Дугласа-Пекера)
                            # epsilon контролирует точность: чем меньше, тем детальнее граница
                            epsilon = 0.0005 * cv2.arcLength(contour, True)
                            approx_contour = cv2.approxPolyDP(contour, epsilon, True)

                            if len(approx_contour) < 3:
                                # YOLO сегментации нужно минимум 3 точки (треугольник)
                                continue

                            # Нормализация координат
                            contour_flat = approx_contour.flatten().astype(float)
                            contour_flat[0::2] /= w  # x
                            contour_flat[1::2] /= h  # y

                            line = f"{class2id[cls_name]} " + " ".join([f"{x:.6f}" for x in contour_flat])
                            yolo_labels.append(line)

                with open(os.path.join(OUT_LABELS, base_name + '.txt'), 'w') as f:
                    f.write("\n".join(yolo_labels))

print("\nПодготовка датасета завершена.")


--------------------------------------------------
Запуск обработки модальности: RGB
--------------------------------------------------

[TRAIN] начинаем обработку...


Обработка train: 100%|██████████| 56944/56944 [59:40<00:00, 15.91it/s]  



[VAL] начинаем обработку...


Обработка val: 100%|██████████| 18334/18334 [18:09<00:00, 16.83it/s] 



[TEST] начинаем обработку...
[TEST] папка с разметкой (labels) не найдена. Будут обработаны только изображения.


...


[VAL] начинаем обработку...


Обработка val: 100%|██████████| 18334/18334 [18:12<00:00, 16.79it/s] 



[TEST] начинаем обработку...
[TEST] папка с разметкой (labels) не найдена. Будут обработаны только изображения.


Обработка test: 100%|██████████| 19708/19708 [06:59<00:00, 46.96it/s]


Подготовка датасета завершена.
